Linear Algebra 2 - Systems of Linear Equations and Orthogonality

Section 13: Systems of Linear Equations
Solving Ax = b via Gaussian elimination and LU decomposition.
Null space and column space. Overdetermined and underdetermined systems.

Section 14: Orthogonality and Gram-Schmidt
Orthogonal vectors, orthonormal bases, Gram-Schmidt process,
orthogonal projection, and PCA whitening.

Section 13 - Systems of Linear Equations

A linear system Ax = b asks: what vector x satisfies A times x equals b?
Three possible outcomes depending on rank:
- unique solution: A is square and full rank
- no solution: b is not in the column space of A (overdetermined)
- infinitely many solutions: A is rank-deficient (underdetermined)

In ML, linear systems appear in the Normal Equation, solving for weights,
and understanding when a model is over or under-specified.

In [ ]:
# Base Python: Gaussian elimination with partial pivoting

def gaussian_solve(A, b):
    n = len(b)
    M = [row[:] + [b[i]] for i, row in enumerate(A)]

    for col in range(n):
        pivot_row = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[pivot_row] = M[pivot_row], M[col]
        pivot = M[col][col]
        for row in range(col + 1, n):
            f = M[row][col] / pivot
            M[row] = [M[row][j] - f * M[col][j] for j in range(n + 1)]

    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        x[i] = M[i][n]
        for j in range(i + 1, n):
            x[i] -= M[i][j] * x[j]
        x[i] /= M[i][i]
    return x

A = [[2.0,  1.0, -1.0],
     [-3.0, -1.0,  2.0],
     [-2.0,  1.0,  2.0]]
b = [8.0, -11.0, -3.0]

x = gaussian_solve(A, b)
print(f'solution x = {[round(v, 4) for v in x]}  (expected [2, 3, -1])')

residual = [sum(A[i][j] * x[j] for j in range(3)) - b[i] for i in range(3)]
print(f'residual   = {[round(r, 8) for r in residual]}  (should be zeros)')

In [ ]:
import numpy as np

A = np.array([[ 2.0,  1.0, -1.0],
              [-3.0, -1.0,  2.0],
              [-2.0,  1.0,  2.0]])
b = np.array([8.0, -11.0, -3.0])

# unique solution (square, full rank)
x = np.linalg.solve(A, b)
print(f'np.linalg.solve: x = {x}')
print(f'verify A @ x = b: {np.allclose(A @ x, b)}')

# overdetermined system -- more equations than unknowns, use least squares
A_over = np.array([[1.0, 1.0],
                   [1.0, 2.0],
                   [1.0, 3.0],
                   [1.0, 4.0]])
b_over = np.array([2.0, 3.0, 5.0, 4.0])

x_ls, residuals, rank, sv = np.linalg.lstsq(A_over, b_over, rcond=None)
print(f'\nleast-squares solution: x = {x_ls.round(4)}')
print(f'singular values: {sv.round(4)}')

In [ ]:
import numpy as np
from scipy.linalg import lu, solve_triangular

A = np.array([[ 2.0,  1.0, -1.0],
              [-3.0, -1.0,  2.0],
              [-2.0,  1.0,  2.0]])
b = np.array([8.0, -11.0, -3.0])

# LU decomposition: P @ A = L @ U
P, L, U = lu(A)
print(f'P:\n{P}')
print(f'L:\n{L.round(4)}')
print(f'U:\n{U.round(4)}')
print(f'P @ A = L @ U: {np.allclose(P @ A, L @ U)}')

# solve via forward then back substitution
Pb = P @ b
y  = solve_triangular(L, Pb, lower=True)   # Ly = Pb
x  = solve_triangular(U, y,  lower=False)  # Ux = y
print(f'\nLU solution: x = {x.round(4)}')

In [ ]:
import torch

A = torch.tensor([[ 2.0,  1.0, -1.0],
                  [-3.0, -1.0,  2.0],
                  [-2.0,  1.0,  2.0]])
b = torch.tensor([8.0, -11.0, -3.0])

x = torch.linalg.solve(A, b)
print(f'PyTorch solve: x = {x}')
print(f'verify: {torch.allclose(A @ x, b)}')

In [ ]:
import tensorflow as tf

A = tf.constant([[ 2.0,  1.0, -1.0],
                 [-3.0, -1.0,  2.0],
                 [-2.0,  1.0,  2.0]])
b = tf.constant([[8.0], [-11.0], [-3.0]])  # shape (3, 1)

x = tf.linalg.solve(A, b)
print(f'TensorFlow solve: x = {x.numpy().flatten().round(4)}')

Null Space and Column Space

The column space of A is the set of all vectors b for which Ax = b has a solution.
It is spanned by the columns of A and has dimension equal to rank(A).

The null space of A is the set of all x where Ax = 0.
It has dimension n - rank(A), called the nullity.

In ML: null space reveals redundant features. If x is in the null space of the
feature matrix X, then adding x to the weight vector changes nothing about predictions.
Regularisation resolves this ambiguity by picking the minimum-norm solution.

In [ ]:
import numpy as np

# rank-deficient matrix: col 3 = col 1 + col 2
A = np.array([[1.0, 2.0,  3.0],
              [4.0, 5.0,  9.0],
              [7.0, 8.0, 15.0]])

print(f'rank(A) = {np.linalg.matrix_rank(A)}  (expected 2)')

U, S, Vt = np.linalg.svd(A)
print(f'singular values: {S.round(6)}')

# null space: rows of Vt corresponding to near-zero singular values
tol = 1e-10
null_vector = Vt[-1]  # last row corresponds to smallest singular value
print(f'null space vector: {null_vector.round(4)}')
print(f'A @ null_vector:   {(A @ null_vector).round(8)}  (should be ~0)')

# column space: first rank(A) columns of U
r = np.linalg.matrix_rank(A)
col_space_basis = U[:, :r]
print(f'\ncolumn space basis shape: {col_space_basis.shape}  (3 dims, rank-2 subspace)')

In [ ]:
import numpy as np

# ML example: two weight vectors giving identical predictions due to null space
np.random.seed(42)
n, d = 100, 3
X = np.random.randn(n, d)
true_w = np.array([2.0, -1.0, 0.5])
y = X @ true_w + 0.1 * np.random.randn(n)

# add bias column
X_b = np.hstack([np.ones((n, 1)), X])
w, _, _, _ = np.linalg.lstsq(X_b, y, rcond=None)
print(f'recovered weights (bias, w1, w2, w3): {w.round(4)}')

# feature matrix with a redundant column: col4 = col1 + col2
X_red = np.hstack([X, X[:, [0]] + X[:, [1]]])
print(f'\nredundant feature matrix rank: {np.linalg.matrix_rank(X_red)}  (expected 3, not 4)')

w1 = np.linalg.lstsq(X_red, y, rcond=None)[0]
_, _, Vt = np.linalg.svd(X_red)
null_dir = Vt[-1]
w2 = w1 + 5.0 * null_dir

print(f'w1 and w2 are different: {not np.allclose(w1, w2)}')
print(f'but predictions are identical: {np.allclose(X_red @ w1, X_red @ w2)}')

Section 14 - Orthogonality and Gram-Schmidt

Two vectors are orthogonal if their dot product is zero.
An orthonormal set has mutually orthogonal unit vectors.

Orthonormal bases are powerful:
- projections reduce to simple dot products
- computations are numerically stable
- the inverse of an orthogonal matrix is just its transpose

Gram-Schmidt converts any linearly independent set into an orthonormal basis
spanning the same subspace, by iteratively removing components already captured.

In [ ]:
# Base Python: Gram-Schmidt orthogonalisation from scratch

def gram_schmidt(vectors):
    basis = []
    for v in vectors:
        w = v[:]
        for q in basis:
            dot = sum(q[i] * v[i] for i in range(len(v)))
            w = [w[i] - dot * q[i] for i in range(len(w))]
        norm = sum(x**2 for x in w) ** 0.5
        basis.append([x / norm for x in w])
    return basis

vectors = [[1.0, 1.0, 0.0],
           [1.0, 0.0, 1.0],
           [0.0, 1.0, 1.0]]

Q = gram_schmidt(vectors)
print('orthonormal basis:')
for i, q in enumerate(Q):
    print(f'  q{i} = {[round(x, 4) for x in q]}')

print('\ndot products between basis vectors (should all be ~0):')
for i in range(len(Q)):
    for j in range(i + 1, len(Q)):
        dp = sum(Q[i][k] * Q[j][k] for k in range(3))
        print(f'  q{i} . q{j} = {dp:.8f}')

print('\nnorms (should all be 1.0):')
for i, q in enumerate(Q):
    print(f'  ||q{i}|| = {sum(x**2 for x in q)**0.5:.6f}')

In [ ]:
import numpy as np

# QR decomposition gives the Gram-Schmidt result directly
# columns of A are the input vectors
A = np.array([[1.0, 1.0, 0.0],
              [1.0, 0.0, 1.0],
              [0.0, 1.0, 1.0]]).T  # columns are our 3 input vectors, shape (3,3)

Q, R = np.linalg.qr(A)
print(f'Q (orthonormal columns):\n{Q.round(4)}')
print(f'\nQ.T @ Q (should be identity):\n{(Q.T @ Q).round(6)}')
print(f'\nreconstruction A = Q @ R: {np.allclose(A, Q @ R)}')

# orthogonal projection of vector b onto the column space of A
b = np.array([3.0, 2.0, 5.0])

# projection matrix: P = Q @ Q.T (since Q has orthonormal columns)
P = Q @ Q.T
b_proj = P @ b
residual = b - b_proj

print(f'\nb = {b}')
print(f'projected b = {b_proj.round(4)}')
print(f'residual    = {residual.round(4)}')
print(f'residual perp to col space: {np.allclose(Q.T @ residual, 0, atol=1e-10)}')

In [ ]:
import torch

A = torch.tensor([[1.0, 1.0, 0.0],
                  [1.0, 0.0, 1.0],
                  [0.0, 1.0, 1.0]]).T

Q, R = torch.linalg.qr(A)
print(f'PyTorch QR:')
print(f'Q shape: {Q.shape}')
print(f'Q.T @ Q = I: {torch.allclose(Q.T @ Q, torch.eye(3), atol=1e-5)}')
print(f'Q @ R = A  : {torch.allclose(Q @ R, A, atol=1e-5)}')

In [ ]:
import tensorflow as tf

A = tf.constant([[1.0, 1.0, 0.0],
                 [1.0, 0.0, 1.0],
                 [0.0, 1.0, 1.0]])
A = tf.transpose(A)

Q, R = tf.linalg.qr(A)
print(f'TensorFlow QR:')
print(f'Q shape: {Q.shape}')
QtQ = tf.transpose(Q) @ Q
print(f'Q.T @ Q = I: {tf.experimental.numpy.allclose(QtQ, tf.eye(3), atol=1e-5)}')

ML Example - PCA Whitening

Whitening transforms data so features are uncorrelated and have unit variance.
This is equivalent to finding an orthonormal basis aligned with the data's principal directions
and then scaling each component to have equal spread.

After whitening, the covariance matrix becomes the identity.
Many learning algorithms converge faster on whitened inputs.
Used in image preprocessing, GMMs, and before PCA for cleaner components.

In [ ]:
import numpy as np

np.random.seed(0)
cov_true = np.array([[4.0, 3.0],
                     [3.0, 4.0]])
X = np.random.multivariate_normal([3.0, 5.0], cov_true, size=500)

# step 1: centre
X_c = X - X.mean(axis=0)

# step 2: covariance
n = X_c.shape[0]
C = (1 / (n - 1)) * X_c.T @ X_c
print(f'original covariance:\n{C.round(4)}')

# step 3: eigendecompose (eigh guarantees real eigenvalues for symmetric matrices)
eigenvalues, eigenvectors = np.linalg.eigh(C)
idx = eigenvalues.argsort()[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]  # columns are orthonormal principal directions

# step 4: PCA whitening -- project then scale
X_proj  = X_c @ eigenvectors              # rotate to principal axes
X_white = X_proj / np.sqrt(eigenvalues)   # scale to unit variance

C_white = (1 / (n - 1)) * X_white.T @ X_white
print(f'\nwhitened covariance (should be ~I):\n{C_white.round(4)}')
print(f'eigenvectors are orthonormal: {np.allclose(eigenvectors.T @ eigenvectors, np.eye(2))}')